In [0]:
# ============================================
# SILVER - ECOMMERCE_CLIENTES
# ============================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

print("✅ Bibliotecas carregadas com sucesso!")

In [0]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)
print("storage_account carregado:", storage_account is not None)

In [0]:
# Configurações OAuth do Service Principal para acesso ao ADLS

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("✅ Configuração do ADLS preparada com sucesso.")

In [0]:
# ==========================================================
# POLLING + LEITURA DA CAMADA BRONZE
# ==========================================================

import time

# ==========================================================
# CAMINHOS
# ==========================================================

# Caminho da camada Bronze
bronze_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "bronze/ecommerce_clientes"
)

# Caminho da camada Silver
silver_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "silver/ecommerce_clientes"
)

# ==========================================================
# POLLING - AGUARDAR A BRONZE FICAR DISPONÍVEL
# ==========================================================

MAX_TENTATIVAS = 3
INTERVALO_SEGUNDOS = 10

for tentativa in range(1, MAX_TENTATIVAS + 1):

    try:
        df_bronze = (
            spark.read
            .format("delta")
            .options(**adls_options)
            .load(bronze_path)
        )

        qtd_registros = df_bronze.count()

        print(
            f"✅ Bronze disponível na tentativa "
            f"{tentativa}. Registros encontrados: {qtd_registros}"
        )

        break

    except Exception as e:

        print(
            f"⏳ Tentativa {tentativa}/{MAX_TENTATIVAS}: "
            "Bronze ainda não disponível."
        )

        if tentativa < MAX_TENTATIVAS:
            print(
                f"Aguardando {INTERVALO_SEGUNDOS} segundos..."
            )
            time.sleep(INTERVALO_SEGUNDOS)

        else:
            raise Exception(
                "❌ Não foi possível acessar a camada Bronze "
                "após todas as tentativas."
            )

print("\n===== CAMINHOS CONFIGURADOS =====")
print("Bronze:", bronze_path)
print("Silver:", silver_path)

In [0]:
# Iniciar transformação Bronze -> Silver

df_silver = df_bronze

print("✅ DataFrame preparado para transformação Silver")
print("Colunas disponíveis:")
print(df_silver.columns)

df_silver.printSchema()

In [0]:
# ============================================
# LIMPEZA E PADRONIZAÇÃO - SILVER
# ============================================

df_silver = (
    df_silver
    .withColumn("nome", F.trim(F.col("nome")))
    .withColumn("sobrenome", F.trim(F.col("sobrenome")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn(
        "dt_cadastro",
        F.col("dt_cadastro").cast("timestamp")
    )
    .withColumn(
        "dt_ultima_atualizacao",
        F.col("dt_ultima_atualizacao").cast("timestamp")
    )
)

print("✅ Limpeza e padronização concluídas")
print("Total de registros:", df_silver.count())

In [0]:
# REGRA 1 - id_cliente não pode ser nulo nem duplicado

ids_nulos = (
    df_silver
    .filter(F.col("id_cliente").isNull())
    .count()
)

ids_duplicados = (
    df_silver
    .groupBy("id_cliente")
    .count()
    .filter(
        (F.col("id_cliente").isNotNull()) &
        (F.col("count") > 1)
    )
    .count()
)

print("===== REGRA 1 - ID_CLIENTE =====")
print("IDs nulos:", ids_nulos)
print("IDs duplicados:", ids_duplicados)

if ids_nulos == 0 and ids_duplicados == 0:
    print("✅ REGRA 1 APROVADA")
else:
    print("❌ REGRA 1 REPROVADA")

In [0]:
# ============================================
# TRATAMENTO REGRA 1
# id_cliente nulo ou duplicado
# ============================================

# 1. Separar registros com id_cliente nulo
df_rejeitados_id = (
    df_silver
    .filter(F.col("id_cliente").isNull())
    .withColumn(
        "motivo_rejeicao",
        F.lit("id_cliente nulo")
    )
)

# 2. Trabalhar somente com IDs não nulos
df_ids_validos = df_silver.filter(
    F.col("id_cliente").isNotNull()
)

# 3. Para IDs duplicados, manter o registro
# com atualização mais recente
window_id = (
    Window
    .partitionBy("id_cliente")
    .orderBy(F.col("dt_ultima_atualizacao").desc_nulls_last())
)

df_ids_validos = (
    df_ids_validos
    .withColumn(
        "_ordem_id",
        F.row_number().over(window_id)
    )
)

# Duplicados que serão rejeitados
df_duplicados_rejeitados = (
    df_ids_validos
    .filter(F.col("_ordem_id") > 1)
    .drop("_ordem_id")
    .withColumn(
        "motivo_rejeicao",
        F.lit("id_cliente duplicado")
    )
)

# Mantém apenas um registro por id_cliente
df_silver = (
    df_ids_validos
    .filter(F.col("_ordem_id") == 1)
    .drop("_ordem_id")
)

print("===== TRATAMENTO REGRA 1 =====")
print("IDs nulos rejeitados:", df_rejeitados_id.count())
print("Duplicados rejeitados:", df_duplicados_rejeitados.count())
print("Registros restantes na Silver:", df_silver.count())

In [0]:
# REGRA 2 - Email deve ser único e ter formato válido

emails_nulos = (
    df_silver
    .filter(F.col("email").isNull())
    .count()
)

emails_duplicados = (
    df_silver
    .filter(F.col("email").isNotNull())
    .groupBy("email")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Regex: deve existir texto antes e depois do @
# e pelo menos um ponto depois do @
regex_email = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

emails_invalidos = (
    df_silver
    .filter(
        F.col("email").isNull() |
        (~F.col("email").rlike(regex_email))
    )
    .count()
)

print("===== REGRA 2 - EMAIL =====")
print("Emails nulos:", emails_nulos)
print("Emails duplicados:", emails_duplicados)
print("Emails com formato inválido:", emails_invalidos)

if emails_duplicados == 0 and emails_invalidos == 0:
    print("✅ REGRA 2 APROVADA")
else:
    print("❌ REGRA 2 REPROVADA")

In [0]:
# ============================================
# TRATAMENTO REGRA 2
# Email nulo, inválido ou duplicado
# ============================================

regex_email = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

# Email nulo
df_email_nulo = (
    df_silver
    .filter(F.col("email").isNull())
    .withColumn(
        "motivo_rejeicao",
        F.lit("email nulo")
    )
)

# Email com formato inválido
df_email_invalido = (
    df_silver
    .filter(
        F.col("email").isNotNull() &
        (~F.col("email").rlike(regex_email))
    )
    .withColumn(
        "motivo_rejeicao",
        F.lit("email com formato invalido")
    )
)

# Primeiro retiramos nulos e inválidos
df_email_validos = (
    df_silver
    .filter(
        F.col("email").isNotNull() &
        F.col("email").rlike(regex_email)
    )
)

# Para email duplicado, mantém o registro mais recente
window_email = (
    Window
    .partitionBy("email")
    .orderBy(F.col("dt_ultima_atualizacao").desc_nulls_last())
)

df_email_validos = (
    df_email_validos
    .withColumn(
        "_ordem_email",
        F.row_number().over(window_email)
    )
)

# Duplicados rejeitados
df_email_duplicado = (
    df_email_validos
    .filter(F.col("_ordem_email") > 1)
    .drop("_ordem_email")
    .withColumn(
        "motivo_rejeicao",
        F.lit("email duplicado")
    )
)

# Mantém somente emails válidos e únicos
df_silver = (
    df_email_validos
    .filter(F.col("_ordem_email") == 1)
    .drop("_ordem_email")
)

print("===== TRATAMENTO REGRA 2 =====")
print("Emails nulos rejeitados:", df_email_nulo.count())
print("Emails inválidos rejeitados:", df_email_invalido.count())
print("Emails duplicados rejeitados:", df_email_duplicado.count())
print("Registros restantes na Silver:", df_silver.count())

In [0]:
# REGRA 3 - Nome e sobrenome não podem ser nulos nem vazios

nomes_invalidos = (
    df_silver
    .filter(
        F.col("nome").isNull() |
        (F.trim(F.col("nome")) == "")
    )
    .count()
)

sobrenomes_invalidos = (
    df_silver
    .filter(
        F.col("sobrenome").isNull() |
        (F.trim(F.col("sobrenome")) == "")
    )
    .count()
)

print("===== REGRA 3 - NOME E SOBRENOME =====")
print("Nomes nulos/vazios:", nomes_invalidos)
print("Sobrenomes nulos/vazios:", sobrenomes_invalidos)

if nomes_invalidos == 0 and sobrenomes_invalidos == 0:
    print("✅ REGRA 3 APROVADA")
else:
    print("❌ REGRA 3 REPROVADA")

In [0]:
# ============================================
# TRATAMENTO REGRA 3
# Nome e sobrenome não podem ser nulos/vazios
# ============================================

# Registros com nome inválido
df_nome_invalido = (
    df_silver
    .filter(
        F.col("nome").isNull() |
        (F.trim(F.col("nome")) == "")
    )
    .withColumn(
        "motivo_rejeicao",
        F.lit("nome nulo ou vazio")
    )
)

# Registros com sobrenome inválido
df_sobrenome_invalido = (
    df_silver
    .filter(
        F.col("sobrenome").isNull() |
        (F.trim(F.col("sobrenome")) == "")
    )
    .withColumn(
        "motivo_rejeicao",
        F.lit("sobrenome nulo ou vazio")
    )
)

# Mantém na Silver somente registros válidos
df_silver = (
    df_silver
    .filter(
        F.col("nome").isNotNull() &
        (F.trim(F.col("nome")) != "") &
        F.col("sobrenome").isNotNull() &
        (F.trim(F.col("sobrenome")) != "")
    )
)

print("===== TRATAMENTO REGRA 3 =====")
print("Nomes inválidos rejeitados:", df_nome_invalido.count())
print("Sobrenomes inválidos rejeitados:", df_sobrenome_invalido.count())
print("Registros restantes na Silver:", df_silver.count())

In [0]:
print("Registros no df_bronze:", df_bronze.count())
print("Registros no df_silver antes da Regra 4:", df_silver.count())

# IDs que existem na Bronze mas não estão mais na Silver
df_ids_removidos = (
    df_bronze
    .select("id_cliente")
    .distinct()
    .join(
        df_silver.select("id_cliente").distinct(),
        on="id_cliente",
        how="left_anti"
    )
)

print("IDs que desapareceram da Silver:")
display(df_ids_removidos)

In [0]:
# REGRA 4 - senha_hash deve ter exatamente 64 caracteres
print("Registros entrando na Regra 4:", df_silver.count())
senhas_nulas = (
    df_silver
    .filter(F.col("senha_hash").isNull())
    .count()
)

senhas_tamanho_invalido = (
    df_silver
    .filter(
        F.col("senha_hash").isNotNull() &
        (F.length(F.col("senha_hash")) != 64)
    )
    .count()
)

print("===== REGRA 4 - SENHA_HASH =====")
print("Senhas nulas:", senhas_nulas)
print("Senhas com tamanho diferente de 64:", senhas_tamanho_invalido)

if senhas_nulas == 0 and senhas_tamanho_invalido == 0:
    print("✅ REGRA 4 APROVADA")
else:
    print("❌ REGRA 4 REPROVADA")

In [0]:
# ============================================================
# TRATAMENTO REGRA 4
# Mantém senha_hash e cria status_senha_hash
# ============================================================

df_silver = df_silver.withColumn(
    "status_senha_hash",
    F.when(
        F.col("senha_hash").isNull(),
        F.lit("INVÁLIDO")
    )
    .when(
        F.length(F.col("senha_hash")) != 64,
        F.lit("INVÁLIDO")
    )
    .otherwise(F.lit("VÁLIDO"))
)

display(
    df_silver.select(
        "id_cliente",
        "senha_hash",
        "status_senha_hash"
    )
)
print("Registros depois do tratamento da Regra 4:", df_silver.count())

In [0]:
df_silver.groupBy("status_senha_hash").count().show()

In [0]:
# REGRA 5 - dt_cadastro não pode ser nula nem futura

datas_nulas = (
    df_silver
    .filter(F.col("dt_cadastro").isNull())
    .count()
)

datas_futuras = (
    df_silver
    .filter(
        F.col("dt_cadastro") > F.current_timestamp()
    )
    .count()
)

print("===== REGRA 5 - DT_CADASTRO =====")
print("Datas nulas:", datas_nulas)
print("Datas futuras:", datas_futuras)

if datas_nulas == 0 and datas_futuras == 0:
    print("✅ REGRA 5 APROVADA")
else:
    print("❌ REGRA 5 REPROVADA")

In [0]:
# ===== TRATAMENTO REGRA 5 =====
# Mantém todos os registros e classifica dt_cadastro

df_silver = df_silver.withColumn(
    "status_dt_cadastro",
    F.when(
        F.col("dt_cadastro").isNull(),
        F.lit("DATA_NULA")
    )
    .when(
        F.col("dt_cadastro") > F.current_timestamp(),
        F.lit("CADASTRO_FUTURO")
    )
    .otherwise(
        F.lit("VALIDO")
    )
)

# Contagens
qtd_validos = df_silver.filter(
    F.col("status_dt_cadastro") == "VALIDO"
).count()

qtd_futuros = df_silver.filter(
    F.col("status_dt_cadastro") == "CADASTRO_FUTURO"
).count()

qtd_nulos = df_silver.filter(
    F.col("status_dt_cadastro") == "DATA_NULA"
).count()

# Resultado
print("===== TRATAMENTO REGRA 5 =====")
print("Registros válidos:", qtd_validos)
print("Cadastros futuros:", qtd_futuros)
print("Datas nulas:", qtd_nulos)
print("Total de registros mantidos:", df_silver.count())

display(
    df_silver.select(
        "id_cliente",
        "dt_cadastro",
        "status_dt_cadastro"
    )
)

In [0]:
df_silver = df_silver.withColumn(
    "status_dt_cadastro",
    F.when(
        F.col("dt_cadastro").isNull(),
        F.lit("DATA_NULA")
    )
    .when(
        F.col("dt_cadastro") > F.current_timestamp(),
        F.lit("CADASTRO_FUTURO")
    )
    .otherwise(
        F.lit("VALIDO")
    )
)

In [0]:
# REGRA 6 - dt_ultima_atualizacao não pode ser anterior a dt_cadastro

atualizacoes_invalidas = (
    df_silver
    .filter(
        F.col("dt_ultima_atualizacao").isNotNull() &
        F.col("dt_cadastro").isNotNull() &
        (F.col("dt_ultima_atualizacao") < F.col("dt_cadastro"))
    )
    .count()
)

print("===== REGRA 6 - DATAS =====")
print("Atualizações anteriores ao cadastro:", atualizacoes_invalidas)

if atualizacoes_invalidas == 0:
    print("✅ REGRA 6 APROVADA")
else:
    print("❌ REGRA 6 REPROVADA")

In [0]:
# ===== TRATAMENTO REGRA 6 =====
# dt_ultima_atualizacao não pode ser anterior a dt_cadastro

df_atualizacao_invalida = (
    df_silver
    .filter(
        F.col("dt_ultima_atualizacao").isNotNull() &
        F.col("dt_cadastro").isNotNull() &
        (F.col("dt_ultima_atualizacao") < F.col("dt_cadastro"))
    )
    .withColumn(
        "motivo_rejeicao",
        F.lit("dt_ultima_atualizacao anterior a dt_cadastro")
    )
)

# Manter somente os registros válidos
df_silver = (
    df_silver
    .filter(
        F.col("dt_ultima_atualizacao").isNull() |
        (F.col("dt_ultima_atualizacao") >= F.col("dt_cadastro"))
    )
)

print("===== TRATAMENTO REGRA 6 =====")
print(
    "Atualizações inválidas rejeitadas:",
    df_atualizacao_invalida.count()
)
print(
    "Registros restantes na Silver:",
    df_silver.count()
)

if df_atualizacao_invalida.count() > 0:
    display(
        df_atualizacao_invalida.select(
            "id_cliente",
            "dt_cadastro",
            "dt_ultima_atualizacao",
            "motivo_rejeicao"
        )
    )

In [0]:
container_squad = "squad1"

def montar_caminho_delta(container, camada, tabela):
    return (
        f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
        f"{camada}/{tabela}"
    )

print("✅ Container:", container_squad)
print("✅ Função montar_caminho_delta criada com sucesso!")

In [0]:
# ===== REGRA 7 - CARREGAR ECOMMERCE_ENDERECOS =====

enderecos_silver_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "silver/ecommerce_enderecos"
)

print("Caminho dos endereços:")
print(enderecos_silver_path)

df_enderecos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(enderecos_silver_path)
)

print("✅ ecommerce_enderecos carregada!")
print("Total de endereços:", df_enderecos.count())
print("Colunas:", df_enderecos.columns)

display(df_enderecos)

In [0]:
# REGRA 7 - Todo cliente deve ter pelo menos 1 endereço associado

# IDs de clientes que possuem endereço
clientes_com_endereco = (
    df_enderecos
    .select("id_cliente")
    .filter(F.col("id_cliente").isNotNull())
    .distinct()
)

# Clientes da Silver que NÃO possuem endereço
df_sem_endereco = (
    df_silver
    .join(
        clientes_com_endereco,
        on="id_cliente",
        how="left_anti"
    )
)

qtd_sem_endereco = df_sem_endereco.count()

print("===== REGRA 7 - CLIENTE COM ENDEREÇO =====")
print("Clientes analisados:", df_silver.count())
print("Clientes sem endereço:", qtd_sem_endereco)

if qtd_sem_endereco == 0:
    print("✅ REGRA 7 APROVADA")
else:
    print("❌ REGRA 7 REPROVADA")

# Mostrar clientes COM seus respectivos endereços

print("Clientes COM endereço:")

display(
    df_silver.alias("c")
    .join(
        df_enderecos.alias("e"),
        F.col("c.id_cliente") == F.col("e.id_cliente"),
        how="inner"
    )
    .select(
        F.col("c.id_cliente").alias("id_cliente"),
        F.col("c.nome").alias("nome"),
        F.col("c.sobrenome").alias("sobrenome"),
        F.col("c.email").alias("email"),
        F.col("e.logradouro").alias("logradouro"),
        F.col("e.numero").alias("numero"),
        F.col("e.complemento").alias("complemento"),
        F.col("e.bairro").alias("bairro"),
        F.col("e.cep").alias("cep"),
        F.col("e.cidade").alias("cidade"),
        F.col("e.estado").alias("estado")
    )
    .orderBy("nome", "sobrenome")
)

In [0]:
# ============================================================
# CÉLULA 25 - CAMINHO ECOMMERCE_PEDIDOS
# ============================================================

pedidos_bronze_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    f"bronze/ecommerce_pedidos"
)

print("📦 Caminho utilizado para a Regra 8:")
print(pedidos_bronze_path)

In [0]:
# ============================================================
# CÉLULA 26 - CARREGAR ECOMMERCE_PEDIDOS DA BRONZE
# ============================================================

from deltalake import DeltaTable

# Opções no formato esperado pelo delta-rs
storage_options_pedidos = {
    "AZURE_STORAGE_ACCOUNT_NAME": storage_account,
    "AZURE_CLIENT_ID": client_id,
    "AZURE_CLIENT_SECRET": client_secret,
    "AZURE_TENANT_ID": tenant_id
}

print("📦 Lendo ecommerce_pedidos da BRONZE...")
print(pedidos_bronze_path)

# Ler a tabela Delta
tabela_pedidos = DeltaTable(
    pedidos_bronze_path,
    storage_options=storage_options_pedidos
)

# Converter para Pandas
pdf_pedidos = tabela_pedidos.to_pandas()

print("✅ ecommerce_pedidos carregado!")
print("Total de registros:", len(pdf_pedidos))
print("Colunas:", list(pdf_pedidos.columns))

display(pdf_pedidos)

In [0]:
# ============================================================
# CÉLULA 27 - REGRA 8
# TODO CLIENTE DEVE TER PELO MENOS 1 PEDIDO
# ============================================================

import pandas as pd

# IDs únicos de clientes que possuem pedidos
ids_com_pedido = set(
    pdf_pedidos["id_cliente"]
    .dropna()
    .astype("int64")
    .unique()
)

# Clientes existentes na Silver
pdf_clientes = df_silver.toPandas()

# Verifica se cada cliente possui pelo menos um pedido
pdf_clientes["tem_pedido"] = (
    pdf_clientes["id_cliente"]
    .astype("int64")
    .isin(ids_com_pedido)
)

# Clientes com pedido
clientes_com_pedido = pdf_clientes[
    pdf_clientes["tem_pedido"] == True
]

# Clientes sem pedido
clientes_sem_pedido = pdf_clientes[
    pdf_clientes["tem_pedido"] == False
]

# Contagens
total_clientes = len(pdf_clientes)
total_com_pedido = len(clientes_com_pedido)
total_sem_pedido = len(clientes_sem_pedido)

print("===== REGRA 8 - CLIENTES COM PEDIDOS =====")
print("Total de clientes:", total_clientes)
print("Clientes com pedido:", total_com_pedido)
print("Clientes sem pedido:", total_sem_pedido)

if total_sem_pedido == 0:
    print("✅ REGRA 8 APROVADA")
else:
    print("❌ REGRA 8 REPROVADA")

print("\n===== CLIENTES SEM PEDIDO =====")
display(clientes_sem_pedido[["id_cliente"]])

In [0]:
# REGRA 9 - Proporção de emails com provedores reconhecidos >= 95%

provedores_validos = [
    "gmail.com",
    "hotmail.com",
    "outlook.com",
    "yahoo.com.br",
    "uol.com.br",
    "live.com"
]

df_regra9 = (
    df_silver
    .withColumn(
        "provedor_email",
        F.lower(F.split(F.col("email"), "@").getItem(1))
    )
)

# Emails fora da lista de provedores reconhecidos
df_email_provedor_invalido = (
    df_regra9
    .filter(~F.col("provedor_email").isin(provedores_validos))
)

total_clientes = df_regra9.count()
total_invalidos = df_email_provedor_invalido.count()
total_validos = total_clientes - total_invalidos

proporcao_validos = (
    (total_validos / total_clientes) * 100
    if total_clientes > 0 else 0
)

print("===== REGRA 9 - PROVEDORES DE EMAIL =====")
print("Total de clientes:", total_clientes)
print("Provedores reconhecidos:", total_validos)
print("Provedores não reconhecidos:", total_invalidos)
print(f"Proporção válida: {proporcao_validos:.2f}%")

if proporcao_validos >= 95:
    print("✅ REGRA 9 APROVADA")
else:
    print("❌ REGRA 9 REPROVADA")

display(
    df_email_provedor_invalido.select(
        "id_cliente",
        "email",
        "provedor_email"
    )
)

In [0]:
# ===== TRATAMENTO REGRA 9 =====
# Criar coluna indicando se o email é válido ou inválido

df_silver = (
    df_silver
    .withColumn(
        "provedor_email",
        F.lower(F.split(F.col("email"), "@").getItem(1))
    )
    .withColumn(
        "status_email",
        F.when(
            F.col("email").isNull(),
            F.lit("INVÁLIDO")
        )
        .when(
            F.col("provedor_email").isin(provedores_validos),
            F.lit("VÁLIDO")
        )
        .otherwise(F.lit("INVÁLIDO"))
    )
)

display(
    df_silver.select(
        "id_cliente",
        "email",
        "provedor_email",
        "status_email"
    )
)

In [0]:
# ============================================================
# REGRA 10 - uuid_cliente deve ser único e formato UUID válido
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Formato padrão UUID
regex_uuid = (
    r"^[0-9a-fA-F]{8}-"
    r"[0-9a-fA-F]{4}-"
    r"[0-9a-fA-F]{4}-"
    r"[0-9a-fA-F]{4}-"
    r"[0-9a-fA-F]{12}$"
)

# Janela para identificar UUID duplicado
window_uuid = Window.partitionBy("uuid_cliente")

df_silver = df_silver.withColumn(
    "status_uuid_cliente",
    F.when(
        F.col("uuid_cliente").isNull() |
        (F.trim(F.col("uuid_cliente")) == "") |
        (~F.col("uuid_cliente").rlike(regex_uuid)) |
        (F.count("*").over(window_uuid) > 1),
        F.lit("INVÁLIDO")
    ).otherwise(
        F.lit("VÁLIDO")
    )
)

# Contagem dos resultados
uuid_validos = (
    df_silver
    .filter(F.col("status_uuid_cliente") == "VÁLIDO")
    .count()
)

uuid_invalidos = (
    df_silver
    .filter(F.col("status_uuid_cliente") == "INVÁLIDO")
    .count()
)

print("===== REGRA 10 - UUID CLIENTE =====")
print("Total de clientes:", df_silver.count())
print("UUIDs válidos:", uuid_validos)
print("UUIDs inválidos:", uuid_invalidos)

if uuid_invalidos == 0:
    print("✅ REGRA 10 APROVADA")
else:
    print("❌ REGRA 10 REPROVADA")

display(
    df_silver.select(
        "id_cliente",
        "uuid_cliente",
        "status_uuid_cliente"
    )
)

In [0]:
# ============================================================
# DATAFRAME FINAL - SILVER ECOMMERCE_CLIENTES
# ============================================================

df_silver_final = df_silver.select(
    "*"
)

print("===== SILVER FINAL =====")
print("Total de registros:", df_silver_final.count())
print("Total de colunas:", len(df_silver_final.columns))

print("\nColunas:")
for coluna in df_silver_final.columns:
    print("-", coluna)

display(df_silver_final)

In [0]:
# ==========================================================
# VALIDAÇÃO ANTES DE SALVAR NA SILVER
# ==========================================================

print("===== VALIDAÇÃO DO DATAFRAME PROCESSADO =====")

print("Total:", df_silver.count())

df_silver.groupBy("status_dt_cadastro") \
    .count() \
    .orderBy("status_dt_cadastro") \
    .show()

print("Datas realmente nulas:")
print(
    df_silver.filter(
        F.col("dt_cadastro").isNull()
    ).count()
)

print("Datas futuras:")
print(
    df_silver.filter(
        F.col("dt_cadastro") > F.current_timestamp()
    ).count()
)

print("Status NULL:")
print(
    df_silver.filter(
        F.col("status_dt_cadastro").isNull()
    ).count()
)

In [0]:
# ===== SALVAR SILVER DE FORMA INCREMENTAL =====

print("Salvando em:")
print(silver_path)

# 1. Garantir que o lote atual não tenha id_cliente duplicado
df_silver_sem_duplicados = (
    df_silver
    .dropDuplicates(["id_cliente"])
)

# 2. Ler registros que já estão salvos na Silver
df_ja_salvos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(silver_path)
)

# 3. Pegar somente IDs que já existem na Silver
ids_ja_salvos = (
    df_ja_salvos
    .select("id_cliente")
    .dropDuplicates(["id_cliente"])
)

# 4. Manter somente registros que ainda NÃO existem na Silver
df_novos_silver = (
    df_silver_sem_duplicados.alias("novo")
    .join(
        ids_ja_salvos.alias("salvo"),
        F.col("novo.id_cliente") == F.col("salvo.id_cliente"),
        "left_anti"
    )
)

# 5. Contagens
qtd_recebidos = df_silver.count()
qtd_sem_duplicados = df_silver_sem_duplicados.count()
qtd_duplicados_removidos = qtd_recebidos - qtd_sem_duplicados
qtd_novos = df_novos_silver.count()

print("Registros recebidos:", qtd_recebidos)
print("Duplicados removidos:", qtd_duplicados_removidos)
print("Novos registros para salvar:", qtd_novos)

# 6. Salvar somente registros realmente novos
if qtd_novos > 0:

    (
        df_novos_silver.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .options(**adls_options)
        .save(silver_path)
    )

    print(f"✅ {qtd_novos} novos registros adicionados à Silver.")

else:
    print("✅ Nenhum registro novo encontrado. Nada foi adicionado.")
    

In [0]:
# ============================================================
# VALIDAÇÃO FINAL DA CAMADA SILVER
# ============================================================

from pyspark.sql import functions as F

# 1. Ler novamente a Silver depois da gravação
df_validacao = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(silver_path)
)

# 2. Total de registros salvos
total = df_validacao.count()

# 3. Verificar IDs duplicados
duplicados = (
    df_validacao
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 4. Verificar id_cliente nulo
ids_nulos = (
    df_validacao
    .filter(F.col("id_cliente").isNull())
    .count()
)

# 5. Verificar status_dt_cadastro nulo
status_null = (
    df_validacao
    .filter(F.col("status_dt_cadastro").isNull())
    .count()
)

# 6. Resultado
print("===== VALIDAÇÃO FINAL DA SILVER =====")
print("Total de registros:", total)
print("IDs duplicados:", duplicados)
print("IDs nulos:", ids_nulos)
print("Status dt_cadastro NULL:", status_null)

if duplicados == 0 and ids_nulos == 0 and status_null == 0:
    print("✅ SILVER VALIDADA COM SUCESSO")
else:
    print("❌ VERIFICAR QUALIDADE DOS DADOS")

In [0]:
# ============================================================
# DIAGNÓSTICO - COMPARAR BRONZE x SILVER
# ============================================================

from pyspark.sql import functions as F

print("=" * 60)
print("🔎 COMPARAÇÃO BRONZE x SILVER")
print("=" * 60)

# Ler Bronze atual
df_bronze_check = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(bronze_path)
)

# Ler Silver atual
df_silver_check = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(silver_path)
)

# Quantidades
total_bronze = df_bronze_check.count()
total_silver = df_silver_check.count()

print(f"🥉 Total Bronze: {total_bronze}")
print(f"🥈 Total Silver: {total_silver}")
print(f"📊 Diferença: {total_silver - total_bronze}")

# IDs que estão na Silver, mas NÃO estão na Bronze
ids_so_silver = (
    df_silver_check
    .select("id_cliente")
    .distinct()
    .join(
        df_bronze_check.select("id_cliente").distinct(),
        on="id_cliente",
        how="left_anti"
    )
)

qtd_so_silver = ids_so_silver.count()

print(f"\n🔍 IDs presentes somente na Silver: {qtd_so_silver}")

if qtd_so_silver > 0:
    print("\n⚠️ Estes IDs estão na Silver, mas não estão na Bronze atual:")
    ids_so_silver.show(100, truncate=False)
else:
    print("\n✅ Não existem IDs extras na Silver.")

# IDs que estão na Bronze, mas NÃO estão na Silver
ids_so_bronze = (
    df_bronze_check
    .select("id_cliente")
    .distinct()
    .join(
        df_silver_check.select("id_cliente").distinct(),
        on="id_cliente",
        how="left_anti"
    )
)

qtd_so_bronze = ids_so_bronze.count()

print(f"\n🔍 IDs presentes na Bronze e ausentes na Silver: {qtd_so_bronze}")

if qtd_so_bronze > 0:
    print("\n⚠️ Estes IDs ainda não chegaram à Silver:")
    ids_so_bronze.show(100, truncate=False)
else:
    print("✅ Todos os IDs da Bronze estão presentes na Silver.")

print("\n" + "=" * 60)
print("🏁 DIAGNÓSTICO FINALIZADO")
print("=" * 60)

In [0]:
# ============================================================
# DIAGNÓSTICO DO ID AUSENTE NA SILVER
# ============================================================

id_investigar = 1790862000

print("=" * 60)
print(f"🔎 INVESTIGANDO ID: {id_investigar}")
print("=" * 60)

print("\n🥉 REGISTRO NA BRONZE:")

(
    df_bronze_check
    .filter(F.col("id_cliente") == id_investigar)
    .show(truncate=False)
)

print("\n📊 Quantidade desse ID na Bronze:")

print(
    df_bronze_check
    .filter(F.col("id_cliente") == id_investigar)
    .count()
)

print("\n🥈 REGISTRO NA SILVER:")

(
    df_silver_check
    .filter(F.col("id_cliente") == id_investigar)
    .show(truncate=False)
)

print("\n🏁 INVESTIGAÇÃO FINALIZADA")